# Stage 6 — Human Push: Five Ideas Past 0.25

**Context:** Concat ties neural on human (0.24 vs 0.25). The neural net is dead weight
on lymph node. Time to attack the features themselves.

**Five ideas (all CPU-only):**
1. **ADT preprocessing ablation** — CLR vs arcsinh vs CLR+per-protein z-score vs robust scaling
2. **UMAP + clustering** — non-linear embedding before clustering
3. **RNA:ADT weight sweep** — find the optimal fusion ratio
4. **ADT feature selection** — top variable proteins only
5. **Ensemble consensus** — consensus over 20 seeds

**Output:** `runs/s6-human-push/`


In [ ]:
import os, sys, json, subprocess

for pkg in ["numpy", "scipy", "pandas", "scikit-learn", "scanpy", "leidenalg", "anndata", "umap-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.preprocessing import StandardScaler, normalize, RobustScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import umap

print("imports ok")


In [ ]:
# ---------- GitHub push ----------
REPO = "/kaggle/working/my_research"
REPO_URL = "https://github.com/rifatahsanpul0k/my_research.git"
def sh(*args):
    r = subprocess.run(list(args), capture_output=True, text=True, cwd=REPO)
    return r
os.makedirs(REPO, exist_ok=True)
OUT = os.path.join(REPO, "runs", "s6-human-push")
os.makedirs(OUT, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    import base64, shutil
    _sec = UserSecretsClient()
    _tok = _sec.get_secret('GITHUB_TOKEN')
    _b64 = base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    _GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {_b64}"]
    if not os.path.isdir(os.path.join(REPO, ".git")):
        if os.listdir(REPO):
            bak = REPO + "_stalebak"
            if os.path.exists(bak):
                shutil.rmtree(bak)
            shutil.move(REPO, bak)
            os.makedirs(REPO, exist_ok=True)
        subprocess.run(_GIT + ["clone", REPO_URL, REPO], check=True)
        sh("git", "config", "user.name", "kaggle-runner")
        sh("git", "config", "user.email", "kaggle-runner@local")
    os.makedirs(OUT, exist_ok=True)
    print("git ready")
except Exception as e:
    print("git setup issue:", str(e)[:200])

def git_push(msg):
    try:
        os.makedirs(OUT, exist_ok=True)
        sh("git", "add", "runs/s6-human-push")
        r = sh("git", "diff", "--cached", "--quiet")
        if r.returncode == 0:
            print("nothing new to push")
            return
        sh("git", "commit", "-m", msg)
        r2 = subprocess.run(_GIT + ["pull", "--rebase", "origin", "main"],
                           capture_output=True, text=True, cwd=REPO)
        r3 = subprocess.run(_GIT + ["push", "origin", "main"],
                           capture_output=True, text=True, cwd=REPO)
        print("pushed")
    except Exception as e:
        print("push failed:", str(e)[:300])


In [ ]:
# ---------- Data loading (EXACT copy from stage5) ----------
MARKER = "10x_human_lymph_node_A1"
def find_data_root():
    for base in ["/kaggle/input/spatial-multiomics-6datasets-private",
                 "/kaggle/working/spatial-multiomics-6datasets-private",
                 "/kaggle/working"]:
        if os.path.isdir(os.path.join(base, MARKER)):
            return base
    return None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    print("dataset not found, attempting download...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d",
                    "pulokpulok/spatial-multiomics-6datasets-private",
                    "-p", "/kaggle/working", "--unzip"], check=True)
    DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

DATASETS = {
    "A1":  dict(folder="10x_human_lymph_node_A1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "D1":  dict(folder="10x_human_lymph_node_D1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "E11": dict(folder="Mouse_Brain_E11_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E13": dict(folder="Mouse_Brain_E13_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E15": dict(folder="Mouse_Brain_E15_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E18": dict(folder="Mouse_Brain_E18_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
}
DECLARED_K = {"A1": 10, "D1": 11, "E11": 8, "E13": 12, "E15": 12, "E18": 14}

def to_dense_small(X, max_elems=5e7):
    if hasattr(X, "toarray"):
        return X.toarray() if X.size <= max_elems else X
    return np.asarray(X)

def load_raw(ds_id):
    """Load RAW features (no normalization yet) — we test normalizations."""
    spec = DATASETS[ds_id]
    root = os.path.join(DATA_ROOT, spec["folder"])
    rna = sc.read_h5ad(os.path.join(root, "adata_RNA.h5ad"))
    rna.var_names_make_unique()
    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, n_top_genes=3000, flavor="seurat", inplace=True)
    Xh = rna[:, rna.var.highly_variable].X
    Xh = Xh.toarray() if hasattr(Xh, "toarray") else np.asarray(Xh)
    Xr = PCA(n_components=64, random_state=0).fit_transform(Xh).astype(np.float32)
    if spec["aux"] == "adt":
        aux = sc.read_h5ad(os.path.join(root, "adata_ADT.h5ad"))
        aux.var_names_make_unique()
        Xa_raw = to_dense_small(aux.X).astype(np.float64)
    else:
        aux = sc.read_h5ad(os.path.join(root, "adata_ATAC.h5ad"))
        aux.var_names_make_unique()
        X = aux.X.tocsr().astype(np.float64) if hasattr(aux.X, "tocsr") else sparse.csr_matrix(np.asarray(aux.X, dtype=np.float64))
        rs = np.asarray(X.sum(axis=1)).ravel() + 1e-9
        tf = sparse.diags(1.0 / rs).dot(X)
        df = np.asarray((X > 0).sum(axis=0)).ravel()
        idf = np.log(1 + X.shape[0] / (df + 1))
        lsi = TruncatedSVD(n_components=50, random_state=0).fit_transform(tf.multiply(idf))
        depth = np.log1p(np.asarray(X.sum(axis=1)).ravel())
        keep = [i for i in range(50) if abs(np.corrcoef(lsi[:, i], depth)[0, 1]) <= 0.75]
        Xa_raw = lsi[:, keep].astype(np.float64)
    gt = pd.read_csv(os.path.join(root, spec["gt_file"]), index_col=0)
    print(f"{ds_id}: RNA {Xr.shape} AUX-raw {Xa_raw.shape}", flush=True)
    return dict(Xr=Xr, Xa_raw=Xa_raw, gt=np.asarray(gt[spec["gt_col"]]).astype(str),
                is_adt=(spec["aux"] == "adt"))


In [ ]:
# ---------- Idea 1: ADT preprocessing variants ----------
def normalize_aux(Xa_raw, method):
    X = np.asarray(Xa_raw, dtype=np.float64)
    if method == "clr":
        gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
        out = np.log1p(X / gm)
    elif method == "arcsinh":
        # arcsinh(x/5) — common ADT alternative, less aggressive than log
        out = np.arcsinh(X / 5.0)
    elif method == "clr_zscore":
        # CLR then per-protein z-score (report's suggested ablation)
        gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
        out = np.log1p(X / gm)
        out = (out - out.mean(axis=0, keepdims=True)) / (out.std(axis=0, keepdims=True) + 1e-8)
    elif method == "robust":
        # arcsinh then per-protein robust scaling (median/IQR)
        out = np.arcsinh(X / 5.0)
        med = np.median(out, axis=0, keepdims=True)
        iqr = np.percentile(out, 75, axis=0, keepdims=True) - np.percentile(out, 25, axis=0, keepdims=True)
        out = (out - med) / (iqr + 1e-8)
    else:
        raise ValueError(method)
    return StandardScaler().fit_transform(out).astype(np.float32)

def cluster(X, head, k, seed=0):
    if head == "kmeans":
        return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
    elif head == "gmm":
        return GaussianMixture(n_components=k, n_init=3, random_state=seed).fit_predict(X)
    elif head == "spherical":
        Xn = normalize(X, norm="l2")
        return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Xn)
    raise ValueError(head)

def eval_config(X, gt, k, heads=("kmeans", "gmm", "spherical"), seeds=(42, 1234, 2024)):
    """Return best mean ARI across heads."""
    best = 0
    best_head = ""
    for head in heads:
        aris = [adjusted_rand_score(gt, cluster(X, head, k, s)) for s in seeds]
        m = float(np.mean(aris))
        if m > best:
            best = m
            best_head = head
    return best, best_head

fout = open(os.path.join(OUT, "human_push_results.jsonl"), "a")
summary = {}


In [ ]:
# ---------- Run all five ideas ----------
NEURAL_BEST = {"A1": 0.250, "D1": 0.250, "E11": 0.445, "E13": 0.436, "E15": 0.506, "E18": 0.323}

for ds_id in DATASETS:
    print(f"\n{'='*60}\nDataset: {ds_id}\n{'='*60}")
    data = load_raw(ds_id)
    Xr_raw, Xa_raw, gt = data["Xr"], data["Xa_raw"], data["gt"]
    k = DECLARED_K[ds_id]
    is_adt = data["is_adt"]
    Xr = StandardScaler().fit_transform(Xr_raw)

    ds_results = {}

    # ---- Idea 1: Preprocessing ablation (ADT only for human, concat for all) ----
    print("\n[Idea 1] Preprocessing variants:")
    pre_methods = ["clr", "arcsinh", "clr_zscore", "robust"] if is_adt else ["standard"]
    for pm in pre_methods:
        Xa = normalize_aux(Xa_raw, pm) if is_adt else StandardScaler().fit_transform(Xa_raw)
        X = StandardScaler().fit_transform(np.concatenate([Xr, Xa], axis=1))
        ari, head = eval_config(X, gt, k)
        ds_results[f"prep_{pm}"] = ari
        fout.write(json.dumps(dict(dataset=ds_id, idea="preprocessing", config=pm,
                                   head=head, ari=ari)) + chr(10))
        fout.flush()
        print(f"  {pm:12} ARI={ari:.3f} ({head})")

    # ---- Idea 2: UMAP + clustering ----
    print("\n[Idea 2] UMAP + clustering:")
    Xa = normalize_aux(Xa_raw, "clr") if is_adt else StandardScaler().fit_transform(Xa_raw)
    X = StandardScaler().fit_transform(np.concatenate([Xr, Xa], axis=1))
    for n_comp in [16, 32]:
        Xu = umap.UMAP(n_components=n_comp, random_state=42, n_neighbors=30,
                       min_dist=0.3).fit_transform(X)
        Xu = StandardScaler().fit_transform(Xu)
        ari, head = eval_config(Xu, gt, k)
        ds_results[f"umap{n_comp}"] = ari
        fout.write(json.dumps(dict(dataset=ds_id, idea="umap", config=f"n_comp={n_comp}",
                                   head=head, ari=ari)) + chr(10))
        fout.flush()
        print(f"  umap-{n_comp:2}d  ARI={ari:.3f} ({head})")

    # ---- Idea 3: RNA:AUX weight sweep ----
    print("\n[Idea 3] Weight sweep:")
    Xa = normalize_aux(Xa_raw, "clr") if is_adt else StandardScaler().fit_transform(Xa_raw)
    for w_rna in [0.0, 0.3, 0.5, 0.7, 1.0]:
        w_aux = 1.0 - w_rna
        X = StandardScaler().fit_transform(
            np.concatenate([Xr * w_rna, Xa * w_aux], axis=1))
        ari, head = eval_config(X, gt, k)
        ds_results[f"w{w_rna:.1f}"] = ari
        fout.write(json.dumps(dict(dataset=ds_id, idea="weight", config=f"rna={w_rna:.1f}",
                                   head=head, ari=ari)) + chr(10))
        fout.flush()
        print(f"  rna={w_rna:.1f}  ARI={ari:.3f} ({head})")

    # ---- Idea 4: AUX feature selection (top variable proteins/components) ----
    print("\n[Idea 4] AUX feature selection:")
    Xa_full = normalize_aux(Xa_raw, "clr") if is_adt else StandardScaler().fit_transform(Xa_raw)
    n_feat = Xa_full.shape[1]
    variances = Xa_full.var(axis=0)
    for frac in [0.25, 0.5, 1.0]:
        n_keep = max(4, int(n_feat * frac))
        top_idx = np.argsort(variances)[-n_keep:]
        Xa = Xa_full[:, top_idx]
        X = StandardScaler().fit_transform(np.concatenate([Xr, Xa], axis=1))
        ari, head = eval_config(X, gt, k)
        ds_results[f"feat{frac}"] = ari
        fout.write(json.dumps(dict(dataset=ds_id, idea="featselect", config=f"frac={frac}",
                                   head=head, ari=ari)) + chr(10))
        fout.flush()
        print(f"  top-{frac*100:.0f}% ({n_keep:3d} feats)  ARI={ari:.3f} ({head})")

    # ---- Idea 5: Ensemble consensus ----
    print("\n[Idea 5] Ensemble consensus:")
    Xa = normalize_aux(Xa_raw, "clr") if is_adt else StandardScaler().fit_transform(Xa_raw)
    X = StandardScaler().fit_transform(np.concatenate([Xr, Xa], axis=1))
    n = X.shape[0]
    coassoc = np.zeros((n, n), dtype=np.float32)
    n_runs = 20
    for s in range(n_runs):
        lab = KMeans(n_clusters=k, n_init=3, random_state=1000 + s).fit_predict(X)
        for c in range(k):
            idx = np.where(lab == c)[0]
            coassoc[np.ix_(idx, idx)] += 1
    coassoc /= n_runs
    # Consensus: cluster the co-association matrix
    cons_lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(coassoc)
    ari = float(adjusted_rand_score(gt, cons_lab))
    ds_results["ensemble"] = ari
    fout.write(json.dumps(dict(dataset=ds_id, idea="ensemble", config="20x kmeans consensus",
                               head="kmeans", ari=ari)) + chr(10))
    fout.flush()
    print(f"  20x consensus  ARI={ari:.3f}")

    # ---- Summary for this dataset ----
    best_cfg = max(ds_results, key=ds_results.get)
    best_ari = ds_results[best_cfg]
    neural = NEURAL_BEST[ds_id]
    gap = neural - best_ari
    verdict = "NEURAL WINS" if gap > 0.02 else ("NEW BEST" if gap < -0.02 else "TIE")
    print(f"\n  Best new idea: {best_cfg} = {best_ari:.3f}")
    print(f"  Neural best: {neural:.3f}  →  {verdict}")
    summary[ds_id] = {"best_idea": best_cfg, "best_ari": float(best_ari),
                      "neural_ari": neural, "verdict": verdict,
                      "all": {kk: float(vv) for kk, vv in ds_results.items()}}

fout.close()

with open(os.path.join(OUT, "human_push_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}\nFINAL SUMMARY\n{'='*60}")
for ds_id, r in summary.items():
    print(f"  {ds_id}: {r['best_idea']}={r['best_ari']:.3f} vs neural={r['neural_ari']:.3f} → {r['verdict']}")

git_push("s6-human-push: five ideas results")
print("\nDone.")
